# Part 4: 딥러닝 모델에 양자화 적용하기


## 1. 환경 설정 및 라이브러리 설치(추천 런타임 유형:T4)


In [1]:
# 라이브러리 설치 (버전 호환성 확보)
# !pip uninstall -y onnxruntime onnxruntime-gpu
!pip install onnx onnxruntime onnxscript numpy==1.26.4

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from copy import deepcopy
import time
import os
import numpy as np
import warnings

# 불필요한 경고 메시지 숨기기
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

PyTorch Version: 2.9.0+cu126
Device: cuda


## 2. 데이터셋 준비 및 기본 모델 훈련 (FP32)

In [3]:
# MNIST 데이터셋 로드
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 기본 CNN 모델 정의
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.reshape(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_model(model, loader, epochs=1):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    return model

def evaluate_model(model, loader, device='cpu'):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def get_model_size(model, path='temp.pth'):
    torch.save(model.state_dict(), path)
    size = os.path.getsize(path)
    os.remove(path)
    return size

print("기본 FP32 모델 훈련 중...")
model_fp32 = SimpleCNN()
model_fp32 = train_model(model_fp32, train_loader)
acc_fp32 = evaluate_model(model_fp32, test_loader, device='cpu')
size_fp32 = get_model_size(model_fp32)
print(f"FP32 Model Accuracy: {acc_fp32:.2f}% | Size: {size_fp32/1024:.2f} KB")

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 476kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.51MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.17MB/s]


기본 FP32 모델 훈련 중...
FP32 Model Accuracy: 98.34% | Size: 1650.03 KB


## 3. PyTorch Dynamic Quantization
가장 간단한 양자화 방법으로, 주로 Linear(FC) 레이어에 적용합니다.

In [4]:
print("PyTorch Dynamic Quantization 적용...")
# Linear 레이어만 동적으로 양자화
model_dynamic = torch.quantization.quantize_dynamic(
    model_fp32.to('cpu'),
    {nn.Linear},
    dtype=torch.qint8
)

acc_dynamic = evaluate_model(model_dynamic, test_loader, device='cpu')
size_dynamic = get_model_size(model_dynamic)
print(f"Dynamic Model Accuracy: {acc_dynamic:.2f}% | Size: {size_dynamic/1024:.2f} KB")

PyTorch Dynamic Quantization 적용...


/tmp/ipython-input-1946685115.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_dynamic = torch.quantization.quantize_dynamic(


Dynamic Model Accuracy: 98.37% | Size: 471.57 KB


## 4. PyTorch Static Quantization
모델의 구조를 수정(QuantStub/DeQuantStub)하고 Calibration 과정을 거쳐 양자화합니다.

- Static Quantization은 실제 추론(inference)에서 사용할 INT8 양자화 모델을 만드는 방식입니다.
- 모델에 **QuantStub / DeQuantStub** 를 추가하여, 입력을 INT8로 바꾸는 구간(Quantize)과 출력에서 다시 FP32로 복원하는 구간(DeQuantize)을 명확히 합니다.
- **Calibration(보정 과정)**: 모델을 변환하기(`convert`) 전에, **실제 데이터(Train Loader)를 모델에 살짝 통과**시킵니다.
    * 이 과정은 학습(Backpropagation)이 아니라, 데이터가 각 레이어를 지날 때 값의 범위(Min-Max)가 어떻게 되는지 관찰하여 양자화 파라미터(Scale, Zero-point)를 확정 짓기 위함입니다.

In [5]:
class QuantizableCNN(nn.Module):
    def __init__(self):
        super(QuantizableCNN, self).__init__()
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.quant(x)
        x = self.pool(self.relu1(self.bn1(self.conv1(x))))
        x = self.pool(self.relu2(self.bn2(self.conv2(x))))
        x = x.reshape(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        x = self.dequant(x)
        return x

    def fuse_model(self):
        torch.quantization.fuse_modules(self, [['conv1', 'bn1', 'relu1'], ['conv2', 'bn2', 'relu2']], inplace=True)

# 1. 새 모델 준비 및 훈련
print("Static Quantization용 모델 훈련 중...")
model_static_base = QuantizableCNN()
model_static_base = train_model(model_static_base, train_loader)
model_static_base.eval()
model_static_base.to('cpu')
model_static_base.fuse_model()

# 2. QConfig 및 Prepare
#   - fbgemm: 메타가 만든 CPU용 고성능 행렬곱(특히 INT8) 라이브러리
model_static_base.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(model_static_base, inplace=True)

# 3. Calibration
print("Calibration 진행 중...")
with torch.no_grad():
    for i, (images, _) in enumerate(train_loader):
        if i > 20: break
        model_static_base(images)

# 4. Convert
torch.quantization.convert(model_static_base, inplace=True)

acc_static = evaluate_model(model_static_base, test_loader, device='cpu')
size_static = get_model_size(model_static_base)
print(f"Static Model Accuracy: {acc_static:.2f}% | Size: {size_static/1024:.2f} KB")

Static Quantization용 모델 훈련 중...


/tmp/ipython-input-698710756.py:40: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.prepare(model_static_base, inplace=True)


Calibration 진행 중...


/tmp/ipython-input-698710756.py:50: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  torch.quantization.convert(model_static_base, inplace=True)


Static Model Accuracy: 98.55% | Size: 423.49 KB


## 5. ONNX Runtime Quantization: ONNX Runtime 양자화
ONNX로 변환 후 Dynamic Quantization을 수행합니다. `ConvInteger` 에러를 방지하기 위해 `op_types_to_quantize`를 설정했습니다.

In [6]:
!pip install onnxscript

In [7]:
print("\n[Step 4] ONNX Runtime Quantization 진행...")
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType, shape_inference

# 경로 설정
os.makedirs('models', exist_ok=True)
onnx_path = "models/mnist_fp32.onnx"
onnx_pre_path = "models/mnist_fp32_preprocessed.onnx" # 전처리된 모델 경로
onnx_quant_path = "models/mnist_int8_dynamic.onnx"

# 1. PyTorch -> ONNX Export
dummy_input = torch.randn(1, 1, 28, 28, device='cpu')
torch.onnx.export(
    model_fp32.to('cpu'),
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
    opset_version=13                  # Reshape 호환성을 위해 13 이상 권장
)
print("1. ONNX Export 완료")

# 2. Shape Inference & Preprocessing
print("2. ONNX Shape Inference(전처리) 수행 중...")
shape_inference.quant_pre_process(
    input_model_path=onnx_path,
    output_model_path=onnx_pre_path,
    skip_symbolic_shape=False
)
print("   - 전처리 완료 (Shape 정보 고정됨)")

# 3. Dynamic Quantization
print("3. 양자화 수행 중...")
quantize_dynamic(
    model_input=onnx_pre_path,              # 전처리된 파일 사용
    model_output=onnx_quant_path,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Gemm'] # Linear 계열만 양자화
)

print(f"ONNX Quantized Model saved at: {onnx_quant_path}")
print(f"ONNX Size: {os.path.getsize(onnx_quant_path)/1024:.2f} KB")

# 4. ONNX Inference Test
def onnx_evaluate(onnx_file, loader):
    # 경고 메시지 끄기 위해 옵션 추가
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    session = ort.InferenceSession(onnx_file, sess_options, providers=['CPUExecutionProvider'])
    input_name = session.get_inputs()[0].name
    correct = 0
    total = 0

    for images, labels in loader:
        outputs = session.run(None, {input_name: images.numpy()})[0]
        predicted = np.argmax(outputs, axis=1)
        total += labels.size(0)
        correct += (predicted == labels.numpy()).sum()
    return 100 * correct / total

acc_onnx = onnx_evaluate(onnx_quant_path, test_loader)
print(f"ONNX Runtime INT8 Accuracy: {acc_onnx:.2f}%")


[Step 4] ONNX Runtime Quantization 진행...


W0122 00:44:36.222000 403 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:68: adapter_lookup: Assertion `false`

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
1. ONNX Export 완료
2. ONNX Shape Inference(전처리) 수행 중...
   - 전처리 완료 (Shape 정보 고정됨)
3. 양자화 수행 중...
ONNX Quantized Model saved at: models/mnist_int8_dynamic.onnx
ONNX Size: 476.70 KB
ONNX Runtime INT8 Accuracy: 98.34%


In [ ]:
def onnx_evaluate(onnx_file, loader):
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    
    session = ort.InferenceSession(onnx_file, sess_options, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    total = 0
    correct = 0
    
    for images, labels in loader:
        outputs = session.run(None, {input_name: images.numpy()})[0]
        total += labels.size(0)
        correct += (predicted == labels.numpy()).sum()
    return 100 * correct / total

acc_onnx = onnx_evaluate(onnx_quant_path, test_loader)
print(f"ONNX Quantized Model Accuracy: {acc_onnx:.2f}%")